### Performance Metrics 

Calculating portfolio performance metrics for each simualtion and putting them into one dataset <br>
Metrics to calculate:
- Final Portfolio value: The final total in the portfolio
- Total Profit: The total amount made from the simulation
- CAGR: The compounded annual growth rate, or the long term growth of the portfolio
- Volatility: the riskiness of the portfolio
- Sharpe Ratio: The return made per unit of risk
- Maximum Drawdown: The largest amount lost from the biggest drop

#### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

#### 2. Load the combined simulations

In [3]:
df = pd.read_csv("../data/simulations/combined_simulations.csv")
df.head()

,Date,SPY_Price,QQQ_Price,VTI_Price,SPY_MA50,QQQ_MA50,VTI_MA50,Position_SPY,Position_QQQ,Position_VTI,...,Cash,Shares_SPY,Shares_QQQ,Shares_VTI,Total_Invested,Portfolio_Value,Earnings,Daily_Return,Strategy,Portfolio_Type
0,2001-06-15,77.995598,35.981022,36.027668,NaN,NaN,NaN,1,0,0,...,0.0,128.212364,0.0,0.0,10000,10000.000000,0.000000,NaN,Buy & Hold,Single
1,2001-06-18,77.617950,35.499580,35.797894,NaN,NaN,NaN,1,0,0,...,0.0,128.212364,0.0,0.0,10000,9951.580934,-48.419066,-0.004842,Buy & Hold,Single
2,2001-06-19,77.957207,35.322208,35.898216,NaN,NaN,NaN,1,0,0,...,0.0,128.212364,0.0,0.0,10000,9995.077785,-4.922215,0.004371,Buy & Hold,Single
3,2001-06-20,78.366867,36.124603,36.276844,NaN,NaN,NaN,1,0,0,...,0.0,128.212364,0.0,0.0,10000,10047.601305,47.601305,0.005255,Buy & Hold,Single
4,2001-06-21,79.256599,36.614502,36.568089,NaN,NaN,NaN,1,0,0,...,0.0,128.212364,0.0,0.0,10000,10161.675995,161.675995,0.011353,Buy & Hold,Single


#### 3. Sort the Data

In [4]:
df = df.sort_values(["Portfolio_Type", "Strategy", "Date"]).copy()

#### 4. Group data by Strategy and Portfolio Type

In [5]:
grouped = df.groupby(["Portfolio_Type", "Strategy"])

#### 5. Calculate Metrics

In [9]:
results = []

for (portfolio_type, strategy), group in grouped:
    
    group = group.sort_values("Date").copy()
    
    print(f"Processing: {portfolio_type} - {strategy}")
    
    #Final Portfolio Value
    final_portfolio_value = group["Portfolio_Value"].iloc[-1]
    
    #Total Profit
    total_invested = group["Total_Invested"].iloc[-1]
    total_profit = final_portfolio_value - total_invested
    
    #CAGR
    initial_portfolio_value = group["Portfolio_Value"].iloc[0]
    years = len(group) / 252   # approximate trading years
    
    if years > 0 and initial_portfolio_value > 0:
        cagr = (final_portfolio_value / initial_portfolio_value) ** (1 / years) - 1
    else:
        cagr = np.nan
        
    #Volatility
    returns = group["Daily_Return"].dropna()
    
    if len(returns) > 1:
        volatility = returns.std() * np.sqrt(252)
    else:
        volatility = np.nan
        
    #Sharpe Ratio, at risk free rate of 0%
    if len(returns) > 1 and returns.std() != 0:
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(252)
    else:
        sharpe_ratio = np.nan
        
    #Max Drawdown
    running_max = group["Portfolio_Value"].cummax()
    drawdown = group["Portfolio_Value"] / running_max - 1
    max_drawdown = drawdown.min()
    
    #append results
    results.append({
        "Portfolio_Type": portfolio_type,
        "Strategy": strategy,
        "CAGR": cagr,
        "Sharpe_Ratio": sharpe_ratio,
        "Volatility": volatility,
        "Max_Drawdown": max_drawdown,
        "Final_Portfolio_Value": final_portfolio_value,
        "Total_Profit": total_profit
    })

Processing: Custom - Buy & Hold
Processing: Custom - Timing
Processing: Equal - Buy & Hold
Processing: Equal - Timing
Processing: Single - Buy & Hold
Processing: Single - Timing


#### 6. Convert to dataframe

In [10]:
summary_metrics_df = pd.DataFrame(results)
summary_metrics_df

,Portfolio_Type,Strategy,CAGR,Sharpe_Ratio,Volatility,Max_Drawdown,Final_Portfolio_Value,Total_Profit
0,Custom,Buy & Hold,0.244609,1.174952,0.204046,-0.462063,2.210672e+06,1.890172e+06
1,Custom,Timing,0.211729,1.428840,0.141459,-0.241306,1.141999e+06,8.214989e+05
2,Equal,Buy & Hold,0.245782,1.173361,0.205356,-0.462449,2.262664e+06,1.942164e+06
3,Equal,Timing,0.212095,1.427044,0.141892,-0.242634,1.150539e+06,8.300388e+05
4,Single,Buy & Hold,0.232282,1.172902,0.194248,-0.468009,1.729331e+06,1.408831e+06
5,Single,Timing,0.189622,1.557310,0.115843,-0.221472,7.250838e+05,4.045838e+05


#### 7. Reformat Columns

In [12]:
#formatting for display
display_df = summary_metrics_df.copy()

display_df["CAGR"] = display_df["CAGR"] * 100
display_df["Volatility"] = display_df["Volatility"] * 100
display_df["Max_Drawdown"] = display_df["Max_Drawdown"] * 100

display_df

,Portfolio_Type,Strategy,CAGR,Sharpe_Ratio,Volatility,Max_Drawdown,Final_Portfolio_Value,Total_Profit
0,Custom,Buy & Hold,24.460917,1.174952,20.404563,-46.206349,2.210672e+06,1.890172e+06
1,Custom,Timing,21.172889,1.428840,14.145918,-24.130585,1.141999e+06,8.214989e+05
2,Equal,Buy & Hold,24.578246,1.173361,20.535595,-46.244903,2.262664e+06,1.942164e+06
3,Equal,Timing,21.209487,1.427044,14.189177,-24.263355,1.150539e+06,8.300388e+05
4,Single,Buy & Hold,23.228226,1.172902,19.424789,-46.800933,1.729331e+06,1.408831e+06
5,Single,Timing,18.962207,1.557310,11.584322,-22.147227,7.250838e+05,4.045838e+05


#### 8. Export dataset

In [14]:
os.makedirs("../data/metrics", exist_ok=True)

summary_metrics_df.to_csv("../data/metrics/summary_metrics.csv", index=False)
print("summary_metrics.csv created successfully")

saved_df = pd.read_csv("../data/metrics/summary_metrics.csv")
saved_df

summary_metrics.csv created successfully


,Portfolio_Type,Strategy,CAGR,Sharpe_Ratio,Volatility,Max_Drawdown,Final_Portfolio_Value,Total_Profit
0,Custom,Buy & Hold,0.244609,1.174952,0.204046,-0.462063,2.210672e+06,1.890172e+06
1,Custom,Timing,0.211729,1.428840,0.141459,-0.241306,1.141999e+06,8.214989e+05
2,Equal,Buy & Hold,0.245782,1.173361,0.205356,-0.462449,2.262664e+06,1.942164e+06
3,Equal,Timing,0.212095,1.427044,0.141892,-0.242634,1.150539e+06,8.300388e+05
4,Single,Buy & Hold,0.232282,1.172902,0.194248,-0.468009,1.729331e+06,1.408831e+06
5,Single,Timing,0.189622,1.557310,0.115843,-0.221472,7.250838e+05,4.045838e+05
